# Optimization and Training Loop

Dans les modules précédents, nous avons construit un réseau de neurones.

Cependant, les poids du réseau sont initialisés aléatoirement.

Le modèle ne sait donc pas encore résoudre le problème.

L'entraînement consiste à ajuster progressivement les paramètres du modèle afin de réduire les erreurs de prédiction.

# Loss Function

La fonction de perte mesure l'écart entre :

- la prédiction du modèle ;
- la vraie réponse.

Plus la loss est faible, plus le modèle est performant.

Supposons :

Vraie classe :

```python
2
```

Prédiction du modèle :

```python
[0.1, 0.2, 0.7]
```

Le modèle attribue la plus forte probabilité à la classe 2.

La loss sera faible.

À l'inverse, une mauvaise prédiction produira une loss plus élevée.

# CrossEntropyLoss

Pour les problèmes de classification, la fonction de perte la plus utilisée est :

```python
nn.CrossEntropyLoss()
```

Elle compare :

- les scores produits par le modèle ;
- les labels réels.

In [ ]:
import torch
from torch import nn

loss_fn = nn.CrossEntropyLoss()

In [ ]:
predictions = torch.tensor(
    [[2.0, 1.0, 0.1]]
)

target = torch.tensor([0])

loss = loss_fn(
    predictions,
    target
)

print(loss)

La prédiction contient :

```python
[2.0, 1.0, 0.1]
```

Le score le plus élevé correspond à la classe :

```python
0
```

Comme la vraie classe est également :

```python
0
```

la loss reste relativement faible.

# Optimizer

L'optimiseur est responsable de la mise à jour des paramètres du modèle.

Après calcul des gradients, il modifie :

- les poids ;
- les biais.

afin de réduire la loss.

PyTorch fournit plusieurs optimiseurs :

- SGD
- Adam
- AdamW
- RMSProp
- Adagrad

Nous commencerons par :

```python
torch.optim.SGD
```

In [ ]:
from torch import optim
model = nn.Linear(
    3,
    2
)
optimizer = optim.SGD(
    model.parameters(),
    lr=0.001
)

Paramètres :

- model.parameters() → paramètres à entraîner ;
- lr → learning rate.

# Learning Rate

Le learning rate contrôle l'amplitude des mises à jour.

Petit learning rate :

Apprentissage lent

Grand learning rate :

Apprentissage rapide mais parfois instable.

lr trop petit
●●●●●●●●

lr adapté
●──●──●──●

lr trop grand
●───────●───────●

# Training Step

Une étape d'entraînement suit toujours le même ordre :

1. Forward Pass
2. Calcul de la Loss
3. Backward Pass
4. Mise à jour des paramètres

In [ ]:
X = torch.rand(
    1,
    3
)
y = torch.tensor([1])
pred = model(X)

loss = loss_fn(
    pred,
    y
)

loss.backward()

optimizer.step()

optimizer.zero_grad()

### loss.backward()

Calcule les gradients.

### optimizer.step()

Met à jour les paramètres.

### optimizer.zero_grad()

Réinitialise les gradients avant l'itération suivante.

# Attention

Dans PyTorch, les gradients s'accumulent.

Sans :

```python
optimizer.zero_grad()
```

les gradients des itérations précédentes seraient ajoutés aux nouveaux gradients.

Cela provoquerait un entraînement incorrect.

In [ ]:
x = torch.tensor(
    2.0,
    requires_grad=True
)

y = x**2

y.backward()

print(x.grad)

y = x**2

y.backward()

print(x.grad)

Le second gradient est ajouté au premier.

C'est pourquoi il faut remettre les gradients à zéro régulièrement.

# Training Loop

Une boucle d'entraînement répète les étapes précédentes sur tous les batches du dataset.

In [ ]:
def train_loop(
    dataloader,
    model,
    loss_fn,
    optimizer
):

    size = len(dataloader.dataset)

    for batch, (X, y) in enumerate(dataloader):

        pred = model(X)

        loss = loss_fn(
            pred,
            y
        )

        loss.backward()

        optimizer.step()

        optimizer.zero_grad()

        if batch % 100 == 0:

            current = batch * len(X)

            print(
                f"loss: {loss.item():>7f} "
                f"[{current:>5d}/{size:>5d}]"
            )

# Evaluation

Pendant l'évaluation :

- on ne calcule pas les gradients ;
- on ne met pas à jour les paramètres.

In [ ]:
def test_loop(
    dataloader,
    model,
    loss_fn
):

    num_batches = len(dataloader)

    test_loss = 0

    correct = 0

    with torch.no_grad():

        for X, y in dataloader:

            pred = model(X)

            test_loss += loss_fn(
                pred,
                y
            ).item()

            correct += (
                pred.argmax(1) == y
            ).type(torch.float).sum().item()

    test_loss /= num_batches

    correct /= len(dataloader.dataset)

    print(
        f"Accuracy: {(100*correct):>0.1f}%"
    )

## Exemple complet

In [ ]:
loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=1e-3
)

In [ ]:
epochs = 5

for t in range(epochs):

    print(
        f"Epoch {t+1}"
    )

    train_loop(
        train_dataloader,
        model,
        loss_fn,
        optimizer
    )

    test_loop(
        test_dataloader,
        model,
        loss_fn
    )

À chaque époque :

- le modèle parcourt toutes les données ;
- les poids sont ajustés ;
- les performances s'améliorent progressivement.